<a href="https://colab.research.google.com/github/annarm1/compling/blob/main/%D0%95%D1%80%D0%BC%D0%BE%D0%BB%D1%8C%D1%87%D0%B5%D0%B2%D0%B0%2C_%D0%9A%D0%BE%D0%B2%D0%B0%D0%BB%D0%B5%D0%BD%D0%BA%D0%BE_%D0%9F%D0%BE%D1%81%D1%82%D1%80%D0%BE%D0%B5%D0%BD%D0%B8%D0%B5_RAG_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D1%8B_%D1%81_%D0%B8%D1%81%D0%BF%D0%BE%D0%BB%D1%8C%D0%B7%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5%D0%BC_LangChain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Тема:** RAG (Retrieval-Augmented Generation) с фреймворком LangChain

Выполнили: Ермольчева Анна, Коваленко Дарья


Разработка RAG-пайплайна


**Задачи:**
* Загрузить набор текстовых документов (например, статей из датасета arXiv Dataset: https://www.kaggle.com/datasets/Cornell-University/arxiv)
* Разбить текст на чанки с помощью Langchain text splitter
* Создать векторный индекс с помощью FAISS и sentence-transformers
* Реализовать langchain-цепочку, которая производим семантический поиск и формирует промпт для LLM (локальной или через Groq/OpenRouter)
* Протестировать систему на нескольких вопросах, оценить качество ответов


**Библиотеки:** langchain, huggingface, faiss-cpu, sentence-transformers

**Ожидаемый результат:** Colab-ноутбук с рабочим прототипом наукоёмкой (например, разработанной на основе текстов ArXiv) RAG-системы, примерами её ответов и качественным анализом, представленным в текстовых блоках


## Загрузить набор текстовых документов

### Датасет с метаданными к статьям

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# путь к файлу датасета
file_path = "arxiv-metadata-oai-snapshot.json"

df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "Cornell-University/arxiv",
  file_path,
  pandas_kwargs={"lines": True, "nrows": 100000}, # указываем число строк
)
df["id"] = df["id"].apply(lambda x: str(x).zfill(9))

In [ ]:
# заглянем
df.head(2)

### Подбор статей

внимание на столбец id: по нему можно добраться до самой статьи

 https://arxiv.org/abs/{id}: посмотреть страницу статьи с абстрактом

 https://arxiv.org/pdf/{id}: скачать



 Выберем только статьи по NLP, чтобы наша модель могла отвечать на вопросы по конкретной тематике, а не всего набора статей, которых не равное количество (по физике, например, статей больше)

In [ ]:
# отфильтруем df по признаку "статьи по NLP" и получим список id для дальнейшего сохранения

ids = df[df["categories"].str.contains("cs.CL", na=False)]["id"].tolist()
print(f'ВСЕГО СТАТЕЙ ПО ТЕМЕ NLP: {len(ids)}')

### Загрузка

In [ ]:
!pip install langchain_community langchain_text_splitters pypdf -q

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# загружаем с PyPDFLoader, используя https://arxiv.org/pdf/{id}
# Некоторые документы в датасете выдают ошибку
# Возможно нет pdf или в id неточность

documents = []

for id in ids:
    try:
        url = f"https://arxiv.org/pdf/{id}"
        loader = PyPDFLoader(url)
        docs = loader.load()
        documents.extend(docs)
        print(f'Загружен {id}')
    except Exception as e:
        print(f"Ошибка с {id}: {e}")

## Разбить на чанки

Был выбран RecursiveCharacterTextSplitter, т.к. в статье указано, что именно этот сплиттер "Best for articles, reports or long documents where maintaining readability and context is crucial.", что подходит под нашу задачу.

Выбираем размер чанка 700 (среднее количество символов на абзац (а в частности - в abstract). Перекрытие 100, чтобы сохранялся контекст с предыдущего чанка

In [ ]:
# выбрать тип text splitter'а под нашу задачу
# например, по статье: https://www.geeksforgeeks.org/artificial-intelligence/text-splitter-in-langchain/

# использовать split_documents
# использовать chunk_size, chunk_overlap для настройки

# сохранить в chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(chunks[0].page_content[:500])  # Пример одного чанка


## Создать векторный индекс с помощью faiss-cpu и sentence-transformers

In [ ]:
!pip install faiss-cpu sentence-transformers langchain-huggingface -q

Выбираем sentence-transformers/all-MiniLM-L6-v2, так как для работы с большими текстами модель подходит

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(chunks, embedding_model)

print(f"Обработано чанков: {vectorstore.index.ntotal}")


## Реализовать цепочку (не удалось по шаблону, взяли другой способ ниже)



In [ ]:
#!pip install langchain_core langchain_classic -q

In [ ]:
#import json
#from langchain_classic.chains import create_retrieval_chain
#from langchain_classic.chains.combine_documents import create_stuff_documents_chain
#from langchain_huggingface import HuggingFaceEndpoint
#from langchain_core.prompts import PromptTemplate

Для генерации ответов выбрали модель mistralai/Mistral-7B-Instruct-v0.2

In [ ]:
# задаем параметры

#GEN_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"
#HF_TOKEN = "" # получить на https://huggingface.co/settings/tokens
#Из-за HF Token блокнот не загружался в репозиторий на гитхаб, поэтому здесь его удалили, но в процессе работы он был получен
# И несмотря на это ошибка далее все равно возникала
#TOP_K = 4 # остановимся на 4х, чтобы не было шумного контекста, как при 8-10 или наоборот нехватки контекста при 1-2 чанках
#QUESTION = "What is the main idea of the article?"
#TASK = "conversational"


In [ ]:
#template = """
#You are a helpful assistant answering questions about NLP papers from arXiv.

#Use ONLY the information from the context below. If the answer is not in the context, say that you don't know and do not guess.

#Question:
#{input}

#Context:
#{context}

#Answer in concise academic English.
#"""

#PROMPT = PromptTemplate.from_template(template)


In [ ]:
#type(PROMPT)

In [ ]:
#retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

#llm = HuggingFaceEndpoint(
    #repo_id=GEN_MODEL_ID,
    #task=TASK,
    #huggingfacehub_api_token=HF_TOKEN,
    #max_new_tokens=256,
    #temperature=0.3,
#)

#question_answer_chain = create_stuff_documents_chain(llm, PROMPT)
#rag_chain = create_retrieval_chain(retriever, question_answer_chain)


При попытке вызвать rag_chain.invoke({"input": QUESTION}) клиент huggingface_hub в Colab выдаёт ошибку несовместимости провайдера и задачи (ValueError: Model ... is not supported for task text-generation and provider featherless-ai. Supported task: conversational). Не получилось ее устранить + эта ошибка не связана, как я понимаю, с логикой RAG, поэтому ниже повторяем шаги RAG-пайплайна вручную, чтобы все равно оценить качество системы

In [ ]:
#def clip_text(text, threshold=100):
    #return f"{text[:threshold]}..." if len(text) > threshold else text

#QUESTION = "What is NLP?"

#resp_dict = rag_chain.invoke({"input": QUESTION})

#clipped_answer = clip_text(resp_dict["answer"], threshold=350)
#print(f"Question:\n{resp_dict['input']}\n\nAnswer:\n{clipped_answer}")
#for i, doc in enumerate(resp_dict["context"]):
    #print()
    #print(f"Source {i + 1}:")
    #print(f"  text: {json.dumps(clip_text(doc.page_content, threshold=350))}")
    #for key in doc.metadata:
        #if key != "pk":
            #val = doc.metadata.get(key)
            #clipped_val = clip_text(val) if isinstance(val, str) else val
            #print(f"  {key}: {clipped_val}")

In [ ]:
#!pip install huggingface_hub -q

#from huggingface_hub import InferenceClient

#client = InferenceClient(
    #model=GEN_MODEL_ID,
    #token=HF_TOKEN,
#)

In [ ]:
#def clip_text(text, threshold=100):
    #return f"{text[:threshold]}..." if len(text) > threshold else text

#def build_prompt(question, docs):
    # docs - список Document
    #context_text = "\n\n".join(doc.page_content for doc in docs)
    # используем тот же шаблон PROMPT
    #return PROMPT.format(input=question, context=context_text)

#def call_llm(prompt_text):
    #completion = client.text_generation(
        #prompt=prompt_text,
        #max_new_tokens=256,
        #temperature=0.3,
    #)
    #return completion


In [ ]:
#QUESTION = "What evaluation metrics are commonly used for neural language models in these papers?"

# 1. ищем релевантные чанки через тот же retriever
#retrieved_docs = retriever.invoke(QUESTION)

# 2. собираем промпт (вопрос + контекст)
#prompt_text = build_prompt(QUESTION, retrieved_docs)

# 3. вызываем LLM через InferenceClient
#answer_text = call_llm(prompt_text)

#print("Question:\n", QUESTION)
#print("\nAnswer:\n", clip_text(answer_text, 350))

#print("\nSOURCES:")
#for i, doc in enumerate(retrieved_docs):
    #print()
    #print(f"Source {i + 1}:")
    #print("  text:", json.dumps(clip_text(doc.page_content, threshold=350)))
    #for key, val in doc.metadata.items():
        #if key != "pk":
            #clipped_val = clip_text(val) if isinstance(val, str) else val
            #print(f"  {key}: {clipped_val}")


# Реализовываем цепочку через OpenRouter

In [ ]:
!pip install langchain-openai -q

In [ ]:
from google.colab import userdata
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
import json

In [ ]:
OPEN_ROUTER_API_KEY = userdata.get('OPEN_ROUTER_KEY')

llm = ChatOpenAI(
    api_key=OPEN_ROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    model="arcee-ai/trinity-large-preview:free",
    temperature=0.3,
    max_tokens=256,
)

In [ ]:
# PROMPT для RAG
template = """
You are a helpful assistant answering questions about NLP papers from arXiv.

Use ONLY the information from the context below. If the answer is not in the context, say that you don't know and do not guess.

Question:
{question}

Context:
{context}

Answer in concise academic English.
"""

PROMPT = PromptTemplate.from_template(template)

In [ ]:
def clip_text(text, threshold=100):
    return f"{text[:threshold]}..." if len(text) > threshold else text

def build_prompt(question, docs):
    context_text = "\n\n".join(doc.page_content for doc in docs)
    return PROMPT.format(question=question, context=context_text)

In [ ]:
TOP_K = 4
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

## Протестировать на нескольких примерах, оченить качество

In [ ]:
QUESTION = "What evaluation metrics are commonly used for neural language models in these papers?"

# 1. ищем релевантные чанки
retrieved_docs = retriever.invoke(QUESTION)

# 2. собираем промпт для LLM
prompt_text = build_prompt(QUESTION, retrieved_docs)

# 3. вызываем LLM через OpenRouter
response = llm.invoke(prompt_text)
answer_text = response.content if hasattr(response, "content") else str(response)

print("Question:\n", QUESTION)
print("\nAnswer:\n", clip_text(answer_text, 350))

print("\nSOURCES:")
for i, doc in enumerate(retrieved_docs):
    print()
    print(f"Source {i + 1}:")
    print("  text:", json.dumps(clip_text(doc.page_content, threshold=350)))
    for key, val in doc.metadata.items():
        if key != "pk":
            clipped_val = clip_text(val) if isinstance(val, str) else val
            print(f"  {key}: {clipped_val}")


In [ ]:
questions = [
    # фактический
    "What evaluation metrics are commonly used for neural language models in these papers?",
    # обобщающий
    "What are the main research directions in recent NLP papers from this corpus?",
    # уточняющий
    "In which contexts do the papers mention transformer architectures and attention mechanisms?",
]

for q in questions:
    print("=" * 80)
    print("QUESTION:", q)

    retrieved_docs = retriever.invoke(q)
    prompt_text = build_prompt(q, retrieved_docs)
    response = llm.invoke(prompt_text)
    answer_text = response.content if hasattr(response, "content") else str(response)

    print("\nANSWER:\n", clip_text(answer_text, 350))

    print("\nSOURCES:")
    for i, doc in enumerate(retrieved_docs):
        print()
        print(f"Source {i + 1}:")
        print("  text:", json.dumps(clip_text(doc.page_content, threshold=350)))
        for key, val in doc.metadata.items():
            if key != "pk":
                clipped_val = clip_text(val) if isinstance(val, str) else val
                print(f"  {key}: {clipped_val}")


**Фактический вопрос** - метрики - Для фактического вопроса про evaluation metrics модель отвечает, что не может назвать конкретные метрики, потому что в контексте не хватает информации. При этом retriever возвращает фрагменты статей 0704.3708 и 0704.3665, где обсуждаются статистики синтаксических структур и методы построения языковых моделей, но явные метрики типа Perplexity, BLEU не упоминаются. Такой ответ можно считать корректным: модель честно следует промпту и не придумывает метрики, которых нет в текстах.

**Обощающий вопрос** - research directions - на это вопрос модель выделяет три темы: dependency grammar и синтаксические сети, изучение детской речи и аннотирование корпусов. Все три направления хорошо соотносятся с содержимым источников, в которых описаны построение синтаксических графов, использование корпуса CHILDES и ручная разметка зависимостей (источники из статьи 0704.3708). Ответ достаточно точно обобщает эти фрагменты, но не охватывает весь корпус cs.CL, поэтому получившаяся «картина исследований» отражает лишь выбранные статьи.

!NB
Для ответа на обобщающий вопрос retriever выбрал чанки из статьи 0704.3708 (страницы 6–8, 18), где обсуждаются dependency grammar, сеть синтаксических связей и использование корпуса CHILDES. Именно эти фрагменты содержат описания основных исследовательских шагов, поэтому модель использует их как основу для обобщения.



**Уточняющий вопрос**- research directions - здесь мы сознательно спрашиваем о трансформерах и attention, которые не встречаются в выбранных статьях. Модель отвечает, что таких упоминаний в контексте нет, что соответствует действительности: источники содержат обсуждение синтаксических сетей, морфологии и интерфейсов ввода текста, но ни трансформеры, ни attention там не фигурируют. Это демонстрирует, что промпт с инструкцией «If the answer is not in the context, say that you don't know» работает: модель не галлюцинирует современные архитектуры, когда их нет в данных.


В целом, на наш взгляд, RAG-система лучше всего справляется с фактическими и уточняющими вопросами: если нужная информация присутствует в чанках, модель корректно её извлекает, а при отсутствии сведений явно отвечает, что не знает. Обобщающие вопросы решаются слабее: ответ отражает содержание нескольких выбранных статей, но не даёт полной картины по всей области NLP, поскольку корпус небольшой и темы статей ограничены. Возможные причины: небольшой размер выборки cs.CL, простая эмбеддинг-модель (all-MiniLM-L6-v2), а также ограничение TOP_K=4, из-за которого в контекст попадает только часть релевантных фрагментов.



В ходе работы мы пробовали подключить LLM через HuggingFaceEndpoint, чтобы использовать модели вроде mistralai/Mistral-7B-Instruct-v0.2. Однако при вызове rag_chain.invoke(...) возникала внутренняя ошибка клиента huggingface_hub (несовместимость задач text-generation и доступного провайдера featherless-ai, поддерживающего только conversational). Настроить Inference API под эти ограничения в рамках учебного проекта не удалось, поэтому для финального прототипа мы выбрали интеграцию через OpenRouter и ChatOpenAI. При этом архитектура RAG-пайплайна сохранилась прежней: семантический поиск по FAISS, формирование промпта на основе найденных чанков и генерация ответа LLM. Таким образом, система остаётся полностью соответствующей заданию (LLM через Groq/OpenRouter), а переход на OpenRouter позволил избежать нестабильности HuggingFace-инференса и сосредоточиться на качестве поиска и ответов.



---



# Критерии оценки

Работа проверяется по следующим критериям (максимум 10 баллов):

### Загрузка и подготовка данных (2 балла)
- [ ] 0.5 балла: выбран критерий подбора материалов
- [ ] 0.5 балла: загружено не менее 100 записей/статей
- [ ] 0.5 балла: тексты успешно извлечены из источника
- [ ] 0.5 балла: данные приведены к формату, пригодному для чанкинга (очистка, объединение полей)

### Чанкинг (2 балла)
- [ ] 0.5 балла: выбран подходящий тип сплиттера (RecursiveCharacterTextSplitter, HTMLHeaderTextSplitter и т.д.)
- [ ] 0.5 балла: обоснован выбор размера чанка и перекрытия (например, "512 токенов, overlap 20% для сохранения контекста")
- [ ] 0.5 балла: чанки созданы и не содержат явных артефактов (оборванных слов)
- [ ] 0.5 балла: количество чанков соответствует ожидаемому (не 1 и не 100500 на документ)

### Векторное хранилище (1 балл)
- [ ] 0.5 балла: выбрана адекватная эмбеддинг-модель (например, all-MiniLM-L6-v2 для русского/английского)
- [ ] 0.5 балла: индекс создан


### Реализация цепочки (3 балла)
- [ ] 0.5 балла: выбрана LLM
- [ ] 1 балл: промпт, QUESTION, TASK составлены корректно
- [ ] 0.5 балла: обоснован заданный TOP_K
- [ ] 0.5 балла: ответ генерируется на основе найденных чанков (видно по содержанию)
- [ ] 0.5 балла: обработан случай отсутствия информации в контексте

### Тестирование и анализ (2 балла)
- [ ] 0.5 балла: задано минимум 3 разнотипных вопроса (фактический, обобщающий, уточняющий)
- [ ] 0.5 балла: для каждого вопроса показан и проанализирован ответ
- [ ] 0.5 балла: в анализе указано, какие чанки использовались и почему
- [ ] 0.5 балла: сделан вывод о качестве работы системы (что получилось, что нет, гипотезы почему)
